In [ ]:
# 標準函式庫
import sys
import os
import warnings
import cv2


# 第三方函式庫 (科學計算/優化)
import emcee
import numpy as np
import scipy.constants as spc
from scipy.interpolate import interp1d
from scipy.optimize import fsolve # 假設你需要 fsolve

# 天文學/數據處理函式庫
from astropy import units as u
from astropy.io import fits 
from astropy.wcs import WCS
from spectral_cube import SpectralCube

# 繪圖函式庫
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import corner
from matplotlib.colors import PowerNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.collections import LineCollection
from matplotlib.ticker import FuncFormatter, FormatStrFormatter
import matplotlib as mpl

# 專案本地模組
import PSSpy as pss

In [ ]:
Local_Standard_Velocity = 5.86 #km/s (Gupta_2024)
pa_deg = 30
pa_rad = pa_deg * np.pi / 180

In [ ]:
ra_start = '19:01:06.961'
ra_end = '19:01:10.248'
dec_deg = 36 + 57/60 + 20/3600  # +31.21.38.180 => 31.3606 度
distance_pc = 160
n_pixels = 789

radius_in_au, radius_out_au = 4.5e2, 1e3

arcsec_range, AU_per_pixel = pss.calc_ra_arcsec(ra_start, ra_end, dec_deg, distance_pc, n_pixels)

print(f"RA arcsec range (rounded): {arcsec_range} arcsec")
print(f"AU per pixel: {AU_per_pixel:.2f} AU/pixel")

In [ ]:

cube = SpectralCube.read('S_CrA_13CO_spw25_tav_jupyter_shifted.fits')
header = fits.getheader('S_CrA_13CO_spw25_tav_jupyter_shifted.fits')
w = WCS(header).sub(['longitude', 'latitude'])

im_center = int(header['CRPIX2']), int(header['CRPIX1'])  # (z, x)
v0 = header['CRVAL3']          # reference velocity
dx = abs(header['CDELT1']) * 3600      # arcsec per pixel
dv = abs(header['CDELT3'])      # km/s per channel
v_lastch_vel = 14.8636
v_lastch_num = 150

# Subcube and moment calculation
velocity_range = [2.4926, 14.8636] * u.km / u.s  # Adjusted velocity range
subcube = cube.spectral_slab(velocity_range[0], velocity_range[1])
moment0 = subcube.moment(order=0).value
moment1 = subcube.moment(order=1).value
rms_channel = 0.026211100061251217
rms_mom0 = 5.134089023394e-2
cube_shape = subcube.shape  # (n_channels, n_z, n_x)

# mom0_3rms = np.ma.masked_where(moment0 < 3.5*rms_channel, moment0)
# mom1_3rms = np.ma.masked_where(moment0 < 3.5*rms_channel, moment1)

In [ ]:
(0.05 / dx) , (0.005 / dv)

In [ ]:
v_weight_for_cube = (dv / dx) ** 2  
max_dist_value = 100


In [ ]:
v_weight_for_cube

In [ ]:
im_center

In [ ]:
hdu_mom0 = fits.PrimaryHDU(data=moment0, header=header)
hdu_mom0.writeto('S_CrA_13CO_mom0.fits', overwrite=True)
hdu_mom1 = fits.PrimaryHDU(data=moment1.data, header=header)
hdu_mom1.header['BUNIT'] = 'km/s'
hdu_mom1.writeto('S_CrA_13CO_mom1.fits', overwrite=True)

In [ ]:
find_streamcom = np.array([[396, 396], [371, 355], [340, 355], [309, 369], [279, 389], [257, 463]]) - (im_center[0], im_center[1])

In [ ]:
find_r, find_theta = pss.spherical_coords(find_streamcom[:, 0], find_streamcom[:, 1])
find_streaml = interp1d(find_r, find_theta)

In [ ]:
center1 = (388, 393)  # (y, x)
center2 = (369, 382)  # (y, x)

new_center = int((center1[0] + center2[0]) / 2), int((center1[1] + center2[1]) / 2) # (y, x)

ny, nx = moment0.shape
radius = 35

# 建立 2D mask 並擴展成 3D mask
mask2d = pss.circular_mask((ny, nx), new_center, radius)
mask3d = np.repeat(mask2d[np.newaxis, :, :], subcube.shape[0], axis=0)

# 套用到 cube
masked_center_cube = subcube.with_mask(mask3d)
# masked_center_cube = masked_center_cube.with_fill_value(0)

In [ ]:
# cube_data = cube.filled_data[:].value  # shape = (z, y, x)
maskcent_cube_data = masked_center_cube.filled_data[:].value  # shape = (z, y, x)

init_points = [(35, new_center[0], new_center[1]), (35, 355, 371), (35, 355, 340), (35, 369, 309), (35, 389, 279), (35, 463, 257)]  # (z, y, x) coordinates of initial points
# stream_mask = pss.grow_region(cube_data, init_points, rms_channel, sigma_thresh=3.5, max_iter=1000)
maskcent_stream_mask = pss.grow_region(maskcent_cube_data, init_points, rms_channel, sigma_thresh=3.5, max_iter=1000)

In [ ]:
masked_cube = masked_center_cube.with_mask(maskcent_stream_mask)

In [ ]:
radius = 11
mask_pos_1 = [320, 412] # (y, x) - (18(ty), 9(tx))

# 建立 2D mask 並擴展成 3D mask
mask2d = pss.circular_mask((ny, nx), mask_pos_1, radius)
mask3d = np.repeat(mask2d[np.newaxis, :, :], subcube.shape[0], axis=0)

# 套用到 cube
masked_cube_1 = masked_cube.with_mask(mask3d)
# masked_center_cube = masked_center_cube.with_fill_value(0)

In [ ]:
radius = 30
mask_pos_2 = [335, 438] # (y, x) - (18(ty), 9(tx))

# 建立 2D mask 並擴展成 3D mask
mask2d = pss.circular_mask((ny, nx), mask_pos_2, radius)
mask3d = np.repeat(mask2d[np.newaxis, :, :], subcube.shape[0], axis=0)

# 套用到 cube
masked_cube_2 = masked_cube_1.with_mask(mask3d)
# masked_center_cube = masked_center_cube.with_fill_value(0)

In [ ]:
# masked_cube_ori = cube.with_mask(stream_mask)
# masked_cube = masked_center_cube.with_mask(maskcent_stream_mask)
new_cube_data = masked_cube_2.with_fill_value(np.nan)
new_cube_data = new_cube_data.filled_data[:].value
# masked_cube = masked_cube.with_fill_value(0)

str_moment0 = masked_cube_2.moment(order=0).value
str_moment1 = masked_cube_2.moment(order=1).value
# str_moment0_ori = masked_cube_ori.moment(order=0).value
# str_moment1_ori = masked_cube_ori.moment(order=1).value

In [ ]:
ty, tx = (im_center[0] - new_center[0], im_center[1] - new_center[1])

# Create the 2x3 translation matrix
# M = [[1, 0, tx],
#      [0, 1, ty]]
M = np.float32([[1, 0, tx], [0, 1, ty]])

# Get the image dimensions
rows, cols= str_moment0.shape

# Apply the affine transformation
shifted_str_mom0 = np.ma.masked_invalid(cv2.warpAffine(str_moment0, M, (cols, rows), borderValue=np.nan)).data
shifted_str_mom1 = np.ma.masked_invalid(cv2.warpAffine(str_moment1, M, (cols, rows), borderValue=np.nan)).data

# shifted_str_mom0_ori = np.ma.masked_invalid(cv2.warpAffine(str_moment0_ori, M, (cols, rows), borderValue=np.nan))
# shifted_str_mom1_ori = np.ma.masked_invalid(cv2.warpAffine(str_moment1_ori, M, (cols, rows), borderValue=np.nan))

In [ ]:
# 獲取平移後的空間維度 (cols=X, rows=Z)
nv, rows, cols = cube_shape

# 初始化一個用於存放平移後數據立方體的陣列
shifted_cube_data = np.full(new_cube_data.shape, np.nan, dtype=new_cube_data.dtype)

# 對 V 軸（速度軸）進行迴圈
for v_slice in range(nv):
    # 提取當前的 2D 空間切片
    original_slice = new_cube_data[v_slice, :, :]
    
    # 應用 2D 仿射變換 (cv2.warpAffine)
    # 使用與 2D 矩陣相同的 M 和 (cols, rows) 尺寸
    shifted_slice = cv2.warpAffine(
        original_slice, 
        M, 
        (cols, rows),  # 圖像尺寸 (X, Z)
        borderValue=np.nan # 邊界填充 NaN
    )
    
    # 將平移後的切片放回新的立方體中
    shifted_cube_data[v_slice, :, :] = shifted_slice



In [ ]:
hdu_mom0 = fits.PrimaryHDU(data=shifted_str_mom0, header=header)
hdu_mom0.writeto('S_CrA_13CO_streamer_mom0.fits', overwrite=True)
hdu_mom1 = fits.PrimaryHDU(data=shifted_str_mom1, header=header)
hdu_mom1.writeto('S_CrA_13CO_streamer_mom1.fits', overwrite=True)

In [ ]:
# # 從 FITS 檔頭獲取必要的 WCS 資訊
# w = WCS(header).sub(['longitude', 'latitude'])
# # 獲取影像的像素維度
# nx = header['NAXIS1']
# ny = header['NAXIS2']

# # 取得影像四個角落的物理座標（以度為單位）
# bottom_left_world = w.pixel_to_world(0, 0)
# top_right_world = w.pixel_to_world(nx - 1, ny - 1)

# # 將度數轉換為相對角秒（以中心點為參考）
# # 這裡我們假設你的 FITS 檔頭已經將參考點 (CRVAL) 設定為影像中心
# # 並且你的座標是從影像中心開始計算的相對座標
# # 如果不是，你需要根據你的中心點重新計算
# ra_offset_arcsec = (bottom_left_world.ra.deg - header['CRVAL1']) * 3600. * np.cos(np.deg2rad(dec_deg))
# dec_offset_arcsec = (bottom_left_world.dec.deg - header['CRVAL2']) * 3600.

# # 設定 extent，確保它是從最小到最大
# xmin = min(ra_offset_arcsec, (top_right_world.ra.deg - header['CRVAL1']) * 3600.)
# xmax = max(ra_offset_arcsec, (top_right_world.ra.deg - header['CRVAL1']) * 3600.)

# ymin = min(dec_offset_arcsec, (top_right_world.dec.deg - header['CRVAL2']) * 3600.)
# ymax = max(dec_offset_arcsec, (top_right_world.dec.deg - header['CRVAL2']) * 3600.)

# extent = (xmax, xmin, ymin, ymax)

# # --- 接下來的繪圖程式碼保持不變 ---
# # 繪圖
# fig = plt.figure(figsize=(8.27, 8.27))
# ax  = fig.add_subplot(111)
# # color
# norm = PowerNorm(gamma=0.5, vmin=0, vmax=np.nanmax(str_moment0))
# imcolor = ax.imshow(str_moment0, origin='lower', cmap='inferno', extent=extent, norm=norm)
# # color bar
# divider = make_axes_locatable(ax)
# cax = divider.append_axes('right', size='3%', pad='1%')
# cbar = fig.colorbar(imcolor, cax=cax)
# cbar.set_label('(Jy/beam km/s)',fontsize=20)
# ax.scatter(0, 0, c='w', s=100, marker='+')
# # range
# ax.set_xlim(12.5, -12.5)
# ax.set_ylim(-12.5, 12.5)

# # axis label
# ax.set_title('S CrA '+r'$\rm ^{13}CO$',fontsize=30)
# ax.set_xlabel('RA Offset (arcsec)',fontsize=20)
# ax.set_ylabel('DEC Offset (arcsec)',fontsize=20)
# plt.show()

In [ ]:
# ----------- input -----------
# parameters for plot
# moment I map in color
cmap       = 'inferno'  # color

# 獲取新中心點的世界坐標 (R.A._ref, Dec._ref)
ref_world = w.pixel_to_world(new_center[1], new_center[0])
ra_ref_deg = ref_world.ra.deg
dec_ref_deg = ref_world.dec.deg

# 獲取影像的像素維度
nx = header['NAXIS1']
ny = header['NAXIS2']

# 定義四個角落的像素坐標 (X 軸)
# 使用 np.array 進行向量化
x_pixels = np.array([0, nx - 1, 0, nx - 1])
y_pixels = np.array([0, 0, ny - 1, ny - 1])

# 透過 WCS 一次性轉換所有角落的坐標 (這是最關鍵的一步)
corner_worlds = w.pixel_to_world(x_pixels, y_pixels)

# 提取 R.A. 和 Dec. 的度數陣列
corner_ra_deg = corner_worlds.ra.deg
corner_dec_deg = corner_worlds.dec.deg

# 計算 R.A. 偏移 (度 -> 角秒)
ra_offset_arcsec = (corner_ra_deg - ra_ref_deg) * 3600. * np.cos(np.deg2rad(dec_deg))

# 計算 Dec 偏移 (度 -> 角秒)
dec_offset_arcsec = (corner_dec_deg - dec_ref_deg) * 3600.

# extent 格式為 (xmin, xmax, ymin, ymax)
# x 軸 (R.A.)
xmin_arcsec = np.min(ra_offset_arcsec)
xmax_arcsec = np.max(ra_offset_arcsec)

# y 軸 (Dec)
ymin_arcsec = np.min(dec_offset_arcsec)
ymax_arcsec = np.max(dec_offset_arcsec)

# 最終的 extent
extent = (xmax_arcsec, xmin_arcsec, ymin_arcsec, ymax_arcsec)

# plot, relative coordinate
# figure
fig = plt.figure(figsize=(8.27, 8.27))
ax  = fig.add_subplot(111)
# color
norm = PowerNorm(gamma=0.5, vmin=0, vmax=np.nanmax(shifted_str_mom0))  # Adjust gamma for better visibility

imcolor = ax.imshow(shifted_str_mom0, origin='lower',
    cmap=cmap, extent=extent, norm=norm)
plt.gca().set_aspect('equal') 

# color bar
divider = make_axes_locatable(ax)
cax     = divider.append_axes('right', size='3%', pad='1%')
cbar = fig.colorbar(imcolor, cax=cax)
cbar.set_label('(Jy/beam km/s)',fontsize=20)

# range
ax.set_xlim(12.5, -12.5)
ax.set_ylim(-12.5, 12.5)

# axis label
ax.set_title('S CrA '+r'$\rm ^{13}CO$',fontsize=30)
ax.set_xlabel('RA Offset (arcsec)',fontsize=20)
ax.set_ylabel('DEC Offset (arcsec)',fontsize=20)

# save/show
folder_path = "image"
outname  = 'S CrA'  # outputname
# plt.savefig(os.path.join(folder_path, outname+'.png'), dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
# -----------------------------------------------------------------------
# 1. 建立 3D 座標網格
# -----------------------------------------------------------------------
# 這裡 v, z, x 是像素編號，與 cube_shape (v, z, x) 順序一致
v, z, x = np.indices(cube_shape)
# 相對於參考像素的座標
x_rel = x - im_center[1]
z_rel = z - im_center[0]

# 計算每個體素的球座標 (r, theta)
# r, theta 會是 3D 陣列，與 data_cube 相同維度
r, theta = pss.spherical_coords(x_rel, z_rel)

N_elements = 10
pars = np.linspace(40, 160, N_elements + 1)  # 徑向距離區間

x_means = np.zeros(N_elements)
z_means = np.zeros(N_elements)
v_means = np.zeros(N_elements)
xzstd = np.zeros(N_elements)

x_array_list = []
z_array_list = []
v_array_list = []
weights_list = []

# -----------------------------------------------------------------------
# 2. 找到每個徑向區間的 xz 平均座標
# -----------------------------------------------------------------------
for i in range(N_elements):
    r_streaml = (pars[i] + pars[i+1]) / 2
    theta0 = find_streaml(r_streaml)

    # 計算 3D 權重陣列
    weight_theta = (x_rel * np.cos(theta0) + z_rel * np.sin(theta0)) / r
    # 處理 r=0 的情況
    weight_theta[r==0] = 0
    
    # ⚠️ 
    # weight_theta[weight_theta < 0.99] = 0

    # 找出指定徑向範圍和有效數據的體素
    dinds = (r > pars[i]) & (r <= pars[i+1]) & np.isfinite(shifted_cube_data) & (shifted_cube_data > 0)    

    # 這裡的邏輯已經是正確的，但為了除錯，我們可以加入打印語句
    if np.sum(dinds) > 0:
        x_means[i] = np.average(x_rel[dinds], weights=shifted_cube_data[dinds] * weight_theta[dinds])
        z_means[i] = np.average(z_rel[dinds], weights=shifted_cube_data[dinds] * weight_theta[dinds])
        xzstd[i] = np.sqrt(np.average((x_rel[dinds] - x_means[i]) ** 2 + (z_rel[dinds] - z_means[i]) ** 2, weights=shifted_cube_data[dinds] * weight_theta[dinds]))
    else:
        x_means[i] = np.nan
        z_means[i] = np.nan
        xzstd[i] = np.nan

# -----------------------------------------------------------------------
# 3. 創建內插函數，用於計算高斯權重
# -----------------------------------------------------------------------
# 篩選掉 NaN 值，確保內插函數正確
valid_means_mask = np.isfinite(x_means)
if np.sum(valid_means_mask) < 2:
    print("數據太稀疏，無法建立內插函數，請檢查數據或參數")
    # 返回一個空的陣列或直接退出
    streamercom_v_pix = np.array([])
    streamercom_z_pix = np.array([])
    streamercom_x_pix = np.array([])
else:
    r_means, theta_means = pss.spherical_coords(x_means[valid_means_mask], z_means[valid_means_mask])
    theta_r = interp1d(r_means, theta_means, fill_value=(theta_means[0], theta_means[-1]), bounds_error=False)
    std_r = interp1d(r_means, xzstd[valid_means_mask], fill_value=(xzstd[0], xzstd[-1]), bounds_error=False)

    # -----------------------------------------------------------------------
    # 4. 計算 v, z, x 的加權平均
    # -----------------------------------------------------------------------
    for i in range(N_elements):
        r_ref = (pars[i] + pars[i+1]) / 2
        
        # 如果徑向區間無效，直接跳過
        if not np.isfinite(x_means[i]):
            x_means[i] = np.nan
            z_means[i] = np.nan
            v_means[i] = np.nan
            continue
            
        theta_ref = theta_r(r_ref)
        std_ref = std_r(r_ref) / r_ref
        
        # 計算 3D 高斯權重
        delta_theta = np.pi - np.abs(np.pi - np.abs(theta - theta_ref))
        weights = shifted_cube_data * pss.gaussian(delta_theta, 0, std_ref)
        
        dinds = (r > pars[i]) & (r <= pars[i+1]) & np.isfinite(shifted_cube_data) & (shifted_cube_data > 0)

        # 存儲每次迴圈的值
        x_array_list.append(x_rel[dinds])
        z_array_list.append(z_rel[dinds])
        v_array_list.append(v[dinds])
        weights_list.append(weights[dinds])
        
        # 計算加權平均
        x_means[i] = np.average(x_rel[dinds], weights=weights[dinds])  
        z_means[i] = np.average(z_rel[dinds], weights=weights[dinds])
        v_means[i] = np.average(v[dinds], weights=weights[dinds])
        print(f"區間 {i}: x_means={x_means[i]}, z_means={z_means[i]}, v_means={v_means[i]}")


x_rotated = x_means * np.cos(pa_rad) + z_means * np.sin(pa_rad)
z_rotated = -x_means * np.sin(pa_rad) + z_means * np.cos(pa_rad)

# 轉換為物理單位
streamercom_x_AU = x_rotated * AU_per_pixel
streamercom_z_AU = z_rotated * AU_per_pixel
streamercom_v_km = v_lastch_vel + (v_means - v_lastch_num) * dv
streamercom_v_LS_km = streamercom_v_km - Local_Standard_Velocity
print(f"最終的像素座標 (x, z):")
print(f"X: {x_means}")
print(f"Z: {z_means}")
print(f"像素中心: {im_center}")

print(f"\n旋轉後的相對像素座標 (x, z):")
print(f"X: {x_rotated}")
print(f"Z: {z_rotated}")

print(f"從 3D 立方體中提取了 {np.sum(np.isfinite(x_means))} 個有效點。")

In [ ]:
pars

In [ ]:
# ----------- input -----------
# parameters for plot
# moment I map in color
cmap       = 'inferno'  # color

# 獲取新中心點的世界坐標 (R.A._ref, Dec._ref)
ref_world = w.pixel_to_world(new_center[1], new_center[0])
ra_ref_deg = ref_world.ra.deg
dec_ref_deg = ref_world.dec.deg

streamercom_world = w.pixel_to_world(x_means + new_center[1], z_means + new_center[0])

streamercom_ra_arcsec = (streamercom_world.ra.deg - ra_ref_deg) * 3600. * np.cos(np.deg2rad(dec_ref_deg))
streamercom_dec_arcsec = (streamercom_world.dec.deg - dec_ref_deg) * 3600.

# --- 3. 計算 shifted_str_mom0 的正確 extent ---
# extent 必須描述 shifted_str_mom0 這個陣列的物理邊界
# 在這個陣列中，im_center 像素點對應的是天空中的 new_center 位置（即圖上的 0,0）
rows, cols = shifted_str_mom0.shape
dx_arcsec = abs(header['CDELT1']) * 3600.
dz_arcsec = abs(header['CDELT2']) * 3600.

# 計算陣列四個角落的像素，相對於 im_center 的偏移量，再轉換為角秒
# RA 偏移 (X 軸)
ra_min_offset = (0 - im_center[1]) * dx_arcsec
ra_max_offset = (cols - im_center[1]) * dx_arcsec
# DEC 偏移 (Z 軸/Y 軸)
dec_min_offset = (0 - im_center[0]) * dz_arcsec
dec_max_offset = (rows - im_center[0]) * dz_arcsec

# extent 格式為 (left, right, bottom, top)
# RA 軸是反的，所以 left 對應 max_offset, right 對應 min_offset
extent = [ra_max_offset, ra_min_offset, dec_min_offset, dec_max_offset]
# plot, relative coordinate
# figure
fig = plt.figure(figsize=(8.27, 8.27))
ax  = fig.add_subplot(111)
# color
norm = PowerNorm(gamma=0.5, vmin=0, vmax=np.nanmax(shifted_str_mom0))  # Adjust gamma for better visibility

imcolor = ax.imshow(shifted_str_mom0, origin='lower',
    cmap=cmap, extent=extent, norm=norm)
plt.gca().set_aspect('equal') 

ax.scatter(0, 0, c='w', s=100, marker='+')
ax.scatter(streamercom_ra_arcsec, streamercom_dec_arcsec,
           c='white', s=50, marker='o', edgecolors='red', linewidths=1.5,
           label='Streamer Centroids')# range

weight_world = w.pixel_to_world(x_array_list[9] + new_center[1], z_array_list[9] + new_center[0])
weight_ra_arcsec = (weight_world.ra.deg - ra_ref_deg) * 3600. * np.cos(np.deg2rad(dec_ref_deg))
weight_dec_arcsec = (weight_world.dec.deg - dec_ref_deg) * 3600.
sc_weights = ax.scatter(weight_ra_arcsec, weight_dec_arcsec, c=weights_list[9], s=1, alpha=0.3, cmap='grey')

# color bar
divider = make_axes_locatable(ax)

# --- MODIFICATION 2: 添加兩個 colorbar ---
# Colorbar 1 (針對背景的 moment 0 圖)
cax1 = divider.append_axes('right', size='3%', pad='1%')
cbar1 = fig.colorbar(imcolor, cax=cax1)
cbar1.set_label('(Jy/beam km/s)',fontsize=20)

# Colorbar 2 (針對您的 scatter 點)
# 我們再次呼叫 append_axes，它會自動將 cax2 附加到 cax1 的右側
cax2 = divider.append_axes('right', size='3%', pad='1%') 
cbar2 = fig.colorbar(sc_weights, cax=cax2) # 傳入 scatter 物件 sc_weights
cbar2.set_label('Velocity (km/s)', fontsize=20) # 為新的 colorbar 命名

# range
ax.set_xlim(12.5, -12.5)
ax.set_ylim(-12.5, 12.5)

# axis label
ax.set_title('S CrA '+r'$\rm ^{13}CO$',fontsize=30)
ax.set_xlabel('RA Offset (arcsec)',fontsize=20)
ax.set_ylabel('DEC Offset (arcsec)',fontsize=20)

# save/show
folder_path = "image"
outname  = 'S CrA'  # outputname
# plt.savefig(os.path.join(folder_path, outname+'.png'), dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
# 繪圖
fig = plt.figure(figsize=(8.27, 8.27))
ax  = fig.add_subplot(111)
# color
cmap = 'coolwarm'
vmin, vmax = Local_Standard_Velocity - 1, Local_Standard_Velocity + 1   # color range
imcolor = ax.imshow(shifted_str_mom1, origin='lower', cmap=cmap, extent=extent, vmin=vmin, vmax=vmax)
# color bar
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='3%', pad='1%')
cbar = fig.colorbar(imcolor, cax=cax)
cbar.set_label('(km/s)',fontsize=20)
ax.scatter(0, 0, c='w', s=100, marker='+')
ax.scatter(streamercom_ra_arcsec, streamercom_dec_arcsec,
           c=streamercom_v_km, cmap=cmap, vmin=vmin, vmax=vmax,
           s=50, marker='o', edgecolors='black', linewidths=1,
           label='Streamer Centroids')# range


ax.set_xlim(12.5, -12.5)
ax.set_ylim(-12.5, 12.5)

# axis label
ax.set_title('S CrA '+r'$\rm ^{13}CO$',fontsize=30)
ax.set_xlabel('RA Offset (arcsec)',fontsize=20)
ax.set_ylabel('DEC Offset (arcsec)',fontsize=20)
plt.show()

In [ ]:
# 從像素座標轉為 WCS 物件
streamercom_rot_world = w.pixel_to_world(x_rotated + new_center[1], z_rotated + new_center[0])

# 計算相對中心的 RA 和 DEC 偏移
streamercom_rot_ra_arcsec = (streamercom_rot_world.ra.deg - header['CRVAL1']) * 3600.
streamercom_rot_dec_arcsec = (streamercom_rot_world.dec.deg - header['CRVAL2']) * 3600.
# 繪圖
fig = plt.figure(figsize=(8.27, 8.27))
ax  = fig.add_subplot(111)
# color
cmap = 'coolwarm'
vmin, vmax = Local_Standard_Velocity - 1, Local_Standard_Velocity + 1   # color range
imcolor = ax.imshow(str_moment1, origin='lower', cmap=cmap, extent=extent, vmin=vmin, vmax=vmax)
# color bar
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='3%', pad='1%')
cbar = fig.colorbar(imcolor, cax=cax)
cbar.set_label('(km/s)',fontsize=20)
ax.scatter(0, 0, c='w', s=100, marker='+')
ax.scatter(streamercom_rot_ra_arcsec, streamercom_rot_dec_arcsec,
           c=streamercom_v_km, cmap=cmap, vmin=vmin, vmax=vmax,
           s=50, marker='o', edgecolors='black', linewidths=1,
           label='Streamer Centroids')# range
ax.set_xlim(12.5, -12.5)
ax.set_ylim(-12.5, 12.5)

# axis label
ax.set_title('S CrA '+r'$\rm ^{13}CO$',fontsize=30)
ax.set_xlabel('RA Offset (arcsec)',fontsize=20)
ax.set_ylabel('DEC Offset (arcsec)',fontsize=20)
plt.show()

In [ ]:
#######Model parameters#######

radius_ref_au = 240 #Disk edge (au)
M_star = 2
weight_v = (dv / dx) ** 2
Omega_ref = pss.Omega_ref(radius_ref_au, M_star) #Keplerian velocity (radian) 1e-13

n_theta, n_phi, n_T_Myr, n_Inclination, n_Omega = 10, 10, 10, 10, 10
theta = np.linspace(0, np.pi / 2, n_theta+2)[1:-1]  # `Theta` 的範圍 [0, π]
phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)  # `Phi` 的範圍 [0, 2π]
T_Myr = np.linspace(0.5, 2, n_T_Myr) * np.pi / Omega_ref / 1e6 / spc.year# `Time` 的範圍 [1e-2, 1e-1]  np.linspace(0.5, 2, n_T_Myr) * np.pi / Omega_ref / 1e6 / spc.year
Inclination = np.linspace(-np.pi / 2, np.pi / 2, n_Inclination+2)[1:-1]  # `Inclination` 的範圍 [-π/2, π/2]
omega = np.linspace(0, 1, n_Omega+1)[1:]
error = np.zeros((n_theta, n_phi, n_T_Myr, n_Inclination, n_Omega))

for i_theta in range(n_theta):
    for i_phi in range(n_phi):
        for i_T_Myr in range(n_T_Myr):
            for i_Inclination in range(n_Inclination):
                for i_Omega in range(n_Omega):
                    error[i_theta, i_phi, i_T_Myr, i_Inclination, i_Omega] = pss.error_function([theta[i_theta], phi[i_phi]], streamercom_x_AU, streamercom_z_AU, streamercom_v_LS_km, 
                                                                                                weight_v, T_Myr[i_T_Myr], omega[i_Omega], Inclination[i_Inclination], M_star)

In [ ]:
min_theta, min_phi, min_T_Myr, min_Inclination, min_Omega = np.unravel_index(np.argmin(error), error.shape)
M_0 = M_star * 2e30 * spc.G / (200 ** 3 * T_Myr[min_T_Myr] * spc.year * 1e6) # kg
M_dot = M_0 / (T_Myr[min_T_Myr] * spc.year * 1e6) / 2e30 #

print(np.shape(error))
print(np.unravel_index(np.argmin(error), error.shape))
print(min_theta, min_phi, min_T_Myr, min_Inclination)
print('Theta = ' + str(np.rad2deg(theta[min_theta])) + ' deg')
print('Phi = ' + str(np.rad2deg(phi[min_phi])) + ' deg')
print('Time = ' + str(T_Myr[min_T_Myr]) + ' Myr')
print('Inclination = ' + str(np.rad2deg(Inclination[min_Inclination])) + ' deg')
print('Omega = '+ str(omega[min_Omega]))
print(M_0)
print(T_Myr[min_T_Myr] * spc.year * 1e6 * 200 / spc.astronomical_unit)
print('Mass accretion rate = ' + str(M_dot))

In [ ]:
T_Myr

In [ ]:
Theta_init = theta[min_theta]
Phi_init = phi[min_phi]
Inclination_init = Inclination[min_Inclination]
T_init = T_Myr[min_T_Myr]
Omega_init = omega[min_Omega]

In [ ]:
# -----------------------------------------------------------------------
# 動態設定 MCMC 的參數範圍 (已加入邊界檢查)
# -----------------------------------------------------------------------
parameter_prior_ranges = {
    # Theta: 確保索引不會小於 0 或大於陣列長度
    'Theta zero': (theta[max(0, min_theta - 1)], theta[min(len(theta) - 1, min_theta + 1)]),
    
    # Phi: 確保索引不會小於 0 或大於陣列長度
    'Phi zero': (phi[max(0, min_phi - 1)], phi[min(len(phi) - 1, min_phi + 1)]),
    
    # Inclination: 確保索引不會小於 0 或大於陣列長度
    'Inclination': (Inclination[max(0, min_Inclination - 1)], Inclination[min(len(Inclination) - 1, min_Inclination + 1)]),    
    
    # Time: 確保索引不會小於 0 或大於陣列長度
    'Time': (T_Myr[max(0, min_T_Myr - 1)], T_Myr[min(len(T_Myr) - 1, min_T_Myr + 1)]),
    
    # Omega: 確保索引不會小於 0 或大於陣列長度
    'Omega': (omega[max(0, min_Omega - 1)], omega[min(len(omega) - 1, min_Omega + 1)]),
}

# 打印新的範圍以供檢查
print("MCMC 參數的新範圍:")
for param, (min_val, max_val) in parameter_prior_ranges.items():
    if param in ["Theta zero", "Phi zero", "Inclination"]:
        print(f"{param:<12s}: ({np.rad2deg(min_val):.2f}, {np.rad2deg(max_val):.2f}) deg")
    elif param == 'Time':
        print(f"{param:<12s}: ({min_val:.4f}, {max_val:.4f}) Myr")
    else:
        print(f"{param:<12s}: ({min_val:.2f}, {max_val:.2f})")

In [ ]:
# # -----------------------------------------------------------------------
# # 動態設定 MCMC 的參數範圍 (已加入邊界檢查)
# # -----------------------------------------------------------------------
# parameter_prior_ranges = {
#     # Theta: 確保索引不會小於 0 或大於陣列長度
#     'Theta0': (theta[0], theta[9]),
    
#     # Phi: 確保索引不會小於 0 或大於陣列長度
#     'Phi0': (phi[0], phi[9]),
    
#     # Inclination: 確保索引不會小於 0 或大於陣列長度
#     'Incl': (Inclination[0], Inclination[9]),

#     # Time: 確保索引不會小於 0 或大於陣列長度
#     'T': (T_Myr[0], T_Myr[9]),

#     # Omega: 確保索引不會小於 0 或大於陣列長度
#     'Omega': (omega[0], omega[9]),
# }

# # 打印新的範圍以供檢查
# print("MCMC 參數的新範圍:")
# for param, (min_val, max_val) in parameter_prior_ranges.items():
#     if param in ['Theta0', 'Phi0', 'Incl']:
#         print(f"{param:<12s}: ({np.rad2deg(min_val):.2f}, {np.rad2deg(max_val):.2f}) deg")
#     elif param == 'T':
#         print(f"{param:<12s}: ({min_val:.4f}, {max_val:.4f}) Myr")
#     else:
#         print(f"{param:<12s}: ({min_val:.2f}, {max_val:.2f})")

In [ ]:
# 1. 建立一個包含所有初始參數的陣列
initial_guess = np.array([Theta_init, Phi_init, Inclination_init, T_init, Omega_init])

ndim = 5 # 參數數量
nwalkers = 20

# 2. 確保每個參數的擾動都以 initial_guess 為中心
initial_pos = np.zeros((nwalkers, ndim))

labels = ["Theta zero", "Phi zero", "Inclination", "Time", "Omega"]


for i in range(ndim):
    key = labels[i]
        
    # 直接獲取 MCMC 所使用的精確先驗範圍 [min, max]
    min_range = parameter_prior_ranges[key][0]
    max_range = parameter_prior_ranges[key][1]
    
    # 最終的初始位置：在精確的先驗範圍內均勻分佈
    initial_pos[:, i] = np.random.uniform(min_range, max_range, nwalkers)

In [ ]:
# ----------------------------------------------------------------------
# 檢查初始位置分佈
# ----------------------------------------------------------------------
print("---------------- MCMC 初始採樣點分佈檢查 (nwalkers={}) ----------------".format(nwalkers))

for i, label in enumerate(labels):
    data = initial_pos[:, i] # 這是 nwalkers 的實際採樣點
    
    # --- 核心修改 ---
    # 不再從 data 中計算 np.min/np.max，
    # 而是直接從您定義的先驗範圍中提取
    min_val_theory = parameter_prior_ranges[label][0]
    max_val_theory = parameter_prior_ranges[label][1]
    # --- 修改結束 ---

    # 針對角度參數，轉換為度數以供檢查
    if label in ["Theta zero", "Phi zero", "Inclination"]:
        # 轉換理論邊界
        min_val_print = np.rad2deg(min_val_theory)
        max_val_print = np.rad2deg(max_val_theory)
        # 仍然計算實際採樣點的中位數
        median_val = np.median(np.rad2deg(data))
        unit = "deg"
    else:
        # 非角度參數
        min_val_print = min_val_theory
        max_val_print = max_val_theory
        # 仍然計算實際採樣點的中位數
        median_val = np.median(data)
        unit = ""
    
    # 打印理論範圍 (Min/Max) 和實際中位數 (Median)
    print(f"[{label:<11s}]: Median (Sample): {median_val:<.6f}, Defined Min: {min_val_print:<.6f}, Defined Max: {max_val_print:<.6f} {unit}")

print("-----------------------------------------------------------------------")

In [ ]:
x_model, y_model, z_model, u_model, v_model, w_model = pss.PSS_model(Theta_init, Phi_init, Inclination_init, T_init, Omega_init, M_star, radius_in_au, radius_out_au, 100)

# 將物理座標轉換為像素座標
x_pix_rotated = x_model / AU_per_pixel
z_pix_rotated = z_model / AU_per_pixel
x_pix = x_pix_rotated * np.cos(pa_rad) - z_pix_rotated * np.sin(pa_rad) + im_center[1]
z_pix = x_pix_rotated * np.sin(pa_rad) + z_pix_rotated * np.cos(pa_rad) + im_center[0]
v_pix = v_lastch_num + ((v_model + Local_Standard_Velocity) - v_lastch_vel) / dv

# 根據所有初始模型線的座標，計算一個包含所有點的邊界框
buffer = 120 # 可以設定一個合適的緩衝值
v_buffer = 20
search_bound = pss.get_bounding_box(x_pix, z_pix, v_pix, buffer, v_buffer,cube_shape)

print(f"MCMC 採樣器的初始邊界框為：{search_bound}")

In [ ]:
# --- 1. 從 search_bound 中提取 8 個頂點的座標 ---
(v_min, v_max), (z_min, z_max), (x_min, x_max) = search_bound

# 定義 8 個頂點的 (x, z, v) 座標
# (請注意，Plotly 的 Y 軸我們用來對應 z_pix, Z 軸對應 v_pix)
vertices_x = [x_min, x_max, x_max, x_min, x_min, x_max, x_max, x_min]
vertices_z = [z_min, z_min, z_max, z_max, z_min, z_min, z_max, z_max]
vertices_v = [v_min, v_min, v_min, v_min, v_max, v_max, v_max, v_max]

# 定義 12 條邊 (用頂點的索引 0-7)
# (0,1), (1,2), (2,3), (3,0) -> 底部
# (4,5), (5,6), (6,7), (7,4) -> 頂部
# (0,4), (1,5), (2,6), (3,7) -> 垂直邊
edges = [
    (0, 1), (1, 2), (2, 3), (3, 0),
    (4, 5), (5, 6), (6, 7), (7, 4),
    (0, 4), (1, 5), (2, 6), (3, 7)
]

# --- 2. 建立用於繪製 12 條線的座標陣列 ---
# 我們在每條線的結尾加上 None，這樣 Plotly 就不會把它們連在一起
lines_x = []
lines_z = []
lines_v = []
for (start_idx, end_idx) in edges:
    lines_x.extend([vertices_x[start_idx], vertices_x[end_idx], None])
    lines_z.extend([vertices_z[start_idx], vertices_z[end_idx], None])
    lines_v.extend([vertices_v[start_idx], vertices_v[end_idx], None])

# --- 3. 建立 Plotly 圖表 ---
fig = go.Figure()

# 追蹤 1: 您計算出的 PSS_model 點
fig.add_trace(go.Scatter3d(
    x=x_pix,
    y=z_pix,  # 將 z_pix 映射到 Y 軸
    z=v_pix,  # 將 v_pix 映射到 Z 軸
    mode='markers',
    marker=dict(size=2, color='blue'),
    name='Model Points'
))

# 追蹤 2: search_bound 的線框
fig.add_trace(go.Scatter3d(
    x=lines_x,
    y=lines_z,
    z=lines_v,
    mode='lines',
    line=dict(color='gray', width=2),
    name='Search Bound'
))

x_axis_range = [30, 510]
y_axis_range = [195, 675]
z_axis_range = [15, 70]
# --- 4. 設定圖表佈局 ---
fig.update_layout(
    title="MCMC 採樣邊界框 (Search Bound)",
    scene=dict(
        xaxis_title="X 像素",
        yaxis_title="Z 像素",
        zaxis_title="V 通道 (Channel)",
        xaxis = dict(nticks=4, range=x_axis_range,),
        yaxis = dict(nticks=4, range=y_axis_range,),
        zaxis = dict(nticks=4, range=z_axis_range,),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=1)
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [ ]:
# MCMC 採樣器初始化
sampler_short = emcee.EnsembleSampler(
    nwalkers, 
    ndim, 
    pss.log_posterior, 
    args=(new_cube_data, search_bound, parameter_prior_ranges, pa_rad, AU_per_pixel, im_center, dv, v_lastch_vel, v_lastch_num, Local_Standard_Velocity, v_weight_for_cube, max_dist_value, M_star, radius_in_au, radius_out_au)
)
Nsteps = 150  # 運行 10000 步
Burnin = 10  # 丟棄前 1000 步

# 運行採樣
print("開始 MCMC 採樣...")
sampler_short.run_mcmc(initial_pos, Nsteps, progress=True)
print("MCMC 採樣完成。")

# # 獲取樣本 (確保 Nsteps > Burnin)
# samples_short = sampler_short.get_chain(discard=Burnin, flat=True)

# # 檢查樣本數量
# print(f"最終樣本數量: {samples_short.shape[0]}")

In [ ]:
# 獲取樣本 (確保 Nsteps > Burnin)
samples_short = sampler_short.get_chain(discard=Burnin, flat=True)

# 檢查樣本數量
print(f"最終樣本數量: {samples_short.shape[0]}")

In [ ]:
ndim = 5

# --- 1. 根本解決方案：預先轉換角度數據 ---
# 這是修正標題的關鍵！
print("正在將 MCMC 採樣結果中的弧度轉換為角度...")
samples_for_plot = np.copy(samples_short)
angle_indices = [0, 1, 2] # 角度參數的索引 ("Theta0", "Phi0", "Inclination")
time_index = 3

for idx in angle_indices:
    samples_for_plot[:, idx] = np.rad2deg(samples_for_plot[:, idx])

# --- 2. 建立新的標籤 (為角度加上單位) ---
new_labels = []
for i, label in enumerate(labels):
    if i in angle_indices:
        new_labels.append(label + " (°)") # 例如 "Theta0 (°)"
    elif i == time_index:
        new_labels.append(label + " (Myr)") # "T (Myr)"
    else:
        new_labels.append(label) # "Omega" 保持不變

# --- 3. 繪製 Corner Plot ---
# 因為我們傳入的是 "samples_for_plot" (角度數據)，
# 所以 show_titles=True 會自動計算出「角度」的標題，
# 並且座標軸也會自動使用「角度」的刻度。
fig = corner.corner(
    samples_for_plot, 
    labels=new_labels, 
    show_titles=True,
    title_fmt='.2f' # 讓所有標題（包括 T）都顯示為小數點後 2 位
)

# --- 4. 後期處理：為 T 和 Omega 客製化軸標籤 (如您所願) ---
print("正在為 T 和 Omega 參數的座標軸添加客製化格式...")
axes = np.array(fig.get_axes()).reshape(ndim, ndim)

# T 的索引
time_idx = 3 
if ndim > time_idx:
    ax_x_time = axes[ndim-1, time_idx] # T 的 X 軸
    ax_y_time = axes[time_idx, 0]     # T 的 Y 軸
    
    # 應用您想要的 "Myr" 格式
    ax_x_time.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:.2f} Myr'))
    ax_y_time.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:.2f} Myr'))

# Omega 的索引
omega_idx = 4 
if ndim > omega_idx:
    ax_x_omega = axes[ndim-1, omega_idx] # Omega 的 X 軸
    ax_y_omega = axes[omega_idx, 0]     # Omega 的 Y 軸
    
    # 應用您想要的 "%.2f" 格式
    ax_x_omega.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    ax_y_omega.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

# 3. 最後顯示您修改過的「一張圖」
print("繪圖完成。")
plt.show()

In [ ]:
# 1. 確保你已經運行了 report_and_get_best_params 函式並得到了最佳參數
# 假設 samples 已經是 MCMC 的結果
# 獲取弧度形式的最佳參數
Theta_short_best, Phi_short_best, Inclination_short_best, T_short_best, Omega_short_best = pss.report_and_get_best_params(samples_short, 99.7) 

# 2. 將所有 log_likelihood 所需的常數打包成一個 tuple，方便傳入
# 確保這些變數在你的環境中是可用的
model_short_args = (new_cube_data, search_bound, AU_per_pixel, im_center, dv, v_lastch_num, v_lastch_vel)

In [ ]:
# --- 重建最佳模型與誤差立方體 ---

# 1. 重新生成模型線的像素座標
x_model, y_model, z_model, u_model, v_model, w_model = pss.PSS_model(
    Theta_short_best, Phi_short_best, Inclination_short_best, T_short_best, Omega_short_best, 
    M_star, radius_in_au, radius_out_au, 80, scale='log', log_power=1.5
)
# x_model, y_model, z_model, u_model, v_model, w_model = pss.PSS_model(
#     Theta_init, Phi_init, Inclination_init, T_init, Omega_init, 
#     M_star, radius_in_au, radius_out_au, 100
# )
# 假設 log_likelihood 的參數都在 model_args 裡
data_cube, search_bound, AU_per_pixel, im_center, dv, v_lastch_num, v_lastch_vel = model_short_args

# 將物理座標轉換為整數像素座標
cube_shape = data_cube.shape

x_best_pix_rotated_short = x_model / AU_per_pixel
z_best_pix_rotated_short = z_model / AU_per_pixel
x_best_pix_short = x_best_pix_rotated_short * np.cos(pa_rad) - z_best_pix_rotated_short * np.sin(pa_rad) + im_center[1]
z_best_pix_short = x_best_pix_rotated_short * np.sin(pa_rad) + z_best_pix_rotated_short * np.cos(pa_rad) + im_center[0]
v_best_pix_short = v_lastch_num + ((v_model - v_lastch_vel + Local_Standard_Velocity) / dv)

In [ ]:
# 呼叫距離立方體函式，並確保使用整數座標
model_line_coords = zip(v_best_pix_short, z_best_pix_short, x_best_pix_short)
final_distance_cube = pss.grow_distance_cube_bounded(cube_shape, 
                                                    model_line_coords, 
                                                    max_dist_value, 
                                                    v_weight_for_cube,
                                                    bound=search_bound # **使用裁剪後的邊界**
                                                )

# Exclude points that are outside your search boundary (-1 values)
valid_mask = final_distance_cube > 0
v_dis_coords, z_dis_coords, x_dis_coords = np.where(valid_mask)
distances = final_distance_cube[valid_mask]
data_values = data_cube[valid_mask]
error_cube = np.zeros_like(final_distance_cube, dtype=np.float32)
error_cube[valid_mask] = distances * data_values

# We'll normalize by the maximum value found in the valid data
max_error = np.nanmax(error_cube[valid_mask])

# Handle the case of zero division if max_error is 0
if max_error > 0:
    normalized_error_cube = error_cube / max_error
else:
    normalized_error_cube = error_cube

In [ ]:
x_axis_range = [100, 600]
y_axis_range = [100, 600]
z_axis_range = [0, 70]

mask_error = (normalized_error_cube > 0)
v_error_coords, z_error_coords, x_error_coords = np.where(mask_error)
data_error = normalized_error_cube[mask_error]

v_data, z_data, x_data = np.where(data_cube > 0)

fig = go.Figure(data=go.Scatter3d(
    x=x_dis_coords, y=z_dis_coords, z=v_dis_coords,
    mode='markers',
    name='Distance Points',
    marker=dict(
        size=1,
        color=distances,
        colorscale='inferno_r',
        opacity=0.1,
        cmax=np.max(distances),
        cmin=0,
        colorbar=dict(
            title="Distance",
            len=0.5,
            x=0.95,
        )
    )
) )

fig.add_trace(go.Scatter3d(
    x=x_best_pix_short, y=z_best_pix_short, z=v_best_pix_short,
    mode='markers',
    marker=dict(
        size=3.5, # 點太小 (size=1) 可能看不出邊緣，建議稍為調大
        symbol='circle',
        
        # 這是正確設定邊緣線條的方式 (使用 line 參數)
        line=dict(
            width=0.1, 
            color='grey' # 將邊緣顏色設定為黑色
        ),
        
        # 點的顏色 (填滿色) 仍使用 v_pix
        color=v_best_pix_short,
        colorscale='RdBu_r',
        opacity=0.8,
        cmax=np.max(v_data),
        cmin=np.min(v_data),
        colorbar=dict(
            title="Model",
            len=0.5,
            x=0.8,
        )
    ),
    name="Best Fit Model"
))

fig.update_layout(
    title='Best Fit Model and Distance Cube Distribution',
    scene=dict(
        xaxis_title='X (pixels)',
        yaxis_title='Z (pixels)',
        zaxis_title='V (channels)',
        xaxis = dict(nticks=4, range=x_axis_range,),
        yaxis = dict(nticks=4, range=y_axis_range,),
        zaxis = dict(nticks=4, range=z_axis_range,),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=1)
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()

fig = go.Figure(data=go.Scatter3d(
    x=x_error_coords, y=z_error_coords, z=v_error_coords,
    mode='markers',
    name='Error Points',
    marker=dict(
        size=1,
        color=data_error,
        colorscale='inferno',
        opacity=0.8,
        cmax=1,
        cmin=0,
        colorbar=dict(
            title="Error (normalized)",
            len=0.5,
            x=0.95,
        )
    )
) )

fig.add_trace(go.Scatter3d(
    x=x_best_pix_short, y=z_best_pix_short, z=v_best_pix_short,
    mode='markers',
    marker=dict(
        size=3.5, # 點太小 (size=1) 可能看不出邊緣，建議稍為調大
        symbol='circle',
        
        # 這是正確設定邊緣線條的方式 (使用 line 參數)
        line=dict(
            width=0.1, 
            color='grey' # 將邊緣顏色設定為黑色
        ),
        
        # 點的顏色 (填滿色) 仍使用 v_pix
        color=v_best_pix_short,
        colorscale='RdBu_r',
        opacity=0.8,
        cmax=np.nanmax(v_data),
        cmin=np.nanmin(v_data),
        colorbar=dict(
            title="Model",
            len=0.5,
            x=0.8,
        )
    ),
    name="Best Fit Model"
))

fig.update_layout(
    title='Best Fit Model and Observation error Distribution',
    scene=dict(
        xaxis_title='X (pixels)',
        yaxis_title='Z (pixels)',
        zaxis_title='V (channels)',
        xaxis = dict(nticks=4, range=x_axis_range,),
        yaxis = dict(nticks=4, range=y_axis_range,),
        zaxis = dict(nticks=4, range=z_axis_range,),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=1)
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()


fig = go.Figure(data=go.Scatter3d(
    x=x_data, y=z_data, z=v_data,
    mode='markers',
    marker=dict(
        size=1,
        color=v_data,
        colorscale='RdBu_r',
        opacity=0.8,
        cmax=np.max(v_data),
        cmin=np.min(v_data),
    ),
    name="Original Data"
) )

fig.add_trace(go.Scatter3d(
    x=x_best_pix_short, y=z_best_pix_short, z=v_best_pix_short,
    mode='markers',
    marker=dict(
        size=3.5, # 點太小 (size=1) 可能看不出邊緣，建議稍為調大
        symbol='circle',
        
        # 這是正確設定邊緣線條的方式 (使用 line 參數)
        line=dict(
            width=0.1, 
            color='grey' # 將邊緣顏色設定為黑色
        ),
        
        # 點的顏色 (填滿色) 仍使用 v_pix
        color=v_best_pix_short,
        colorscale='RdBu_r',
        opacity=0.8,
        cmax=np.max(v_data),
        cmin=np.min(v_data),
        colorbar=dict(
            title="Model & Observations",
            len=0.5,
            x=0.8,
        )
    ),
    name="Best Fit Model"
))

fig.update_layout(
    title='Best Fit Model and Observations Distribution',
    scene=dict(
        xaxis_title='X (pixels)',
        yaxis_title='Z (pixels)',
        zaxis_title='V (channels)',
        xaxis = dict(nticks=4, range=x_axis_range,),
        yaxis = dict(nticks=4, range=y_axis_range,),
        zaxis = dict(nticks=4, range=z_axis_range,),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=1)
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()

In [ ]:
# 從像素座標轉為 WCS 物件
streamerfitcom_world = w.pixel_to_world(x_best_pix_short, z_best_pix_short)

# 計算相對中心的 RA 和 DEC 偏移
streamerfitcom_ra_arcsec = (streamerfitcom_world.ra.deg - header['CRVAL1']) * 3600.
streamerfitcom_dec_arcsec = (streamerfitcom_world.dec.deg - header['CRVAL2']) * 3600.

# figure
fig = plt.figure(figsize=(8.27, 8.27))
ax  = fig.add_subplot(111)
# color
norm = PowerNorm(gamma=0.5, vmin=0, vmax=np.nanmax(shifted_str_mom0))  # Adjust gamma for better visibility

imcolor = ax.imshow(shifted_str_mom0, origin='lower',
    cmap='inferno', extent=extent, norm=norm)
plt.gca().set_aspect('equal') 

ax.scatter(0, 0, c='w', s=100, marker='+')
# ax.scatter(streamercom_ra_arcsec, streamercom_dec_arcsec,
#            c='white', s=50, marker='o', edgecolors='red', linewidths=1.5,
#            label='Streamer Centroids')# range
points = np.array([streamerfitcom_ra_arcsec, streamerfitcom_dec_arcsec]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
# 黑色邊框線（稍寬一點蓋底）
lc_edge = LineCollection(segments, colors='black', linewidth=10, zorder=1)
ax.add_collection(lc_edge)

# 建立 colormap 的 LineCollection
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
lc = LineCollection(segments, cmap='coolwarm', norm=norm, zorder=2)
lc.set_array(v_model + Local_Standard_Velocity)  # 對應每段顏色
lc.set_linewidth(8)         # 主線寬度
ax.add_collection(lc)       # 加到圖上
ax.add_collection(lc)
ax.add_collection(lc)
# color bar
divider = make_axes_locatable(ax)

# --- MODIFICATION 2: 添加兩個 colorbar ---
# Colorbar 1 (針對背景的 moment 0 圖)
cax1 = divider.append_axes('right', size='3%', pad='1%')
cbar1 = fig.colorbar(imcolor, cax=cax1)
cbar1.set_label('(Jy/beam km/s)',fontsize=20)

# range
ax.set_xlim(12.5, -12.5)
ax.set_ylim(-12.5, 12.5)

# axis label
ax.set_title('S CrA '+r'$\rm ^{13}CO$',fontsize=30)
ax.set_xlabel('RA Offset (arcsec)',fontsize=20)
ax.set_ylabel('DEC Offset (arcsec)',fontsize=20)

# save/show
folder_path = "image"
outname  = 'S CrA'  # outputname
# plt.savefig(os.path.join(folder_path, outname+'.png'), dpi=300, bbox_inches='tight')

plt.show()

In [ ]:

# 繪圖
fig = plt.figure(figsize=(8.27, 8.27))
ax  = fig.add_subplot(111)
# color
cmap = 'coolwarm'
vmin, vmax = Local_Standard_Velocity - 1, Local_Standard_Velocity + 1   # color range
imcolor = ax.imshow(shifted_str_mom1, origin='lower', cmap=cmap, extent=extent, vmin=vmin, vmax=vmax)
# color bar
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='3%', pad='1%')
cbar = fig.colorbar(imcolor, cax=cax)
cbar.set_label('(km/s)',fontsize=20)
ax.scatter(0, 0, c='w', s=100, marker='+')
# ax.scatter(streamercom_ra_arcsec, streamercom_dec_arcsec,
#            c=streamercom_v_km, cmap=cmap, vmin=vmin, vmax=vmax,
#            s=50, marker='o', edgecolors='black', linewidths=1,
#            label='Streamer Centroids')# range
# 黑色邊框線（稍寬一點蓋底）
lc_edge = LineCollection(segments, colors='black', linewidth=10, zorder=1)
ax.add_collection(lc_edge)

# 建立 colormap 的 LineCollection
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
lc = LineCollection(segments, cmap='coolwarm', norm=norm, zorder=2)
lc.set_array(v_model + Local_Standard_Velocity)  # 對應每段顏色
lc.set_linewidth(8)         # 主線寬度
ax.add_collection(lc)       # 加到圖上
ax.add_collection(lc)
ax.add_collection(lc) 
ax.set_xlim(12.5, -12.5)
ax.set_ylim(-12.5, 12.5)

# axis label
ax.set_title('S CrA '+r'$\rm ^{13}CO$',fontsize=30)
ax.set_xlabel('RA Offset (arcsec)',fontsize=20)
ax.set_ylabel('DEC Offset (arcsec)',fontsize=20)
plt.show()